In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'

BASE = '/content/drive/MyDrive/adp_project'
for d in ('data', 'models', 'logs'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip -q install kaggle
!kaggle datasets files solesensei/solesensei_bdd100k

In [ ]:
import os, json, glob, random, shutil, collections, csv
from PIL import Image

BASE = "/content/drive/MyDrive/adp_project"
RAW, OUT = "/content/bdd_raw", "/content/bdd_subset"
assert os.path.isdir("/content/drive/MyDrive"), "Drive not mounted - run the mount cell first"
shutil.rmtree(OUT, ignore_errors=True)

NAMES = ["person","rider","car","truck","bus","train","motor","bike","traffic light","traffic sign"]
MAP = {n: i for i, n in enumerate(NAMES)}
MAP.update({"motorcycle": 6, "bicycle": 7})

def pairs(rs, sh, cap):
    imgs = {os.path.basename(p)[:-4]: p for p in glob.glob(f"{RAW}/data/images_100k/100k/{rs}/{sh}/*.jpg")}
    labs = {os.path.basename(p)[:-5]: p for p in glob.glob(f"{RAW}/data/labels/100k/{rs}/{sh}/*.json")}
    ok = []
    for s in sorted(set(imgs) & set(labs)):
        try:
            with Image.open(imgs[s]) as im:
                im.verify()
            ok.append((s, imgs[s], labs[s]))
        except Exception:
            pass
    random.Random(42).shuffle(ok)
    print(f"{rs}/{sh}: images {len(imgs)}, labels {len(labs)}, complete pairs {len(ok)}")
    return ok[:cap]

S = {"train": pairs("train", "shard_000", 4000),
     "val":   pairs("train", "shard_001", 800),
     "test":  pairs("val",   "shard_000", 800)}

fallback = False
if len(S["test"]) < 300:
    fallback = True
    held = S["train"][-400:]
    S["train"] = S["train"][:-400]
    S["test"] = S["test"] + held
    print("WARNING: test split was small, so 400 unused train images were held out as extra test data")
assert len(S["train"]) > 500 and len(S["val"]) > 100, f"Too few images: {({k: len(v) for k, v in S.items()})}"

unknown, tiny, meta, summary = collections.Counter(), 0, [], {}
for sp, items in S.items():
    os.makedirs(f"{OUT}/images/{sp}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{sp}", exist_ok=True)
    cls, boxes, empty = collections.Counter(), 0, 0
    for s, ip, lp in items:
        shutil.copy(ip, f"{OUT}/images/{sp}/{s}.jpg")
        d = json.load(open(lp, encoding="utf-8"))
        frames = d.get("frames") or [{}]
        lines = []
        for o in frames[0].get("objects", []):
            if "box2d" not in o:
                continue
            c = o.get("category")
            if c not in MAP:
                unknown[c] += 1
                continue
            b = o["box2d"]
            x1, y1 = max(0.0, b["x1"]), max(0.0, b["y1"])
            x2, y2 = min(1280.0, b["x2"]), min(720.0, b["y2"])
            if x2 - x1 <= 1 or y2 - y1 <= 1:
                tiny += 1
                continue
            lines.append(f"{MAP[c]} {(x1+x2)/2/1280:.6f} {(y1+y2)/2/720:.6f} {(x2-x1)/1280:.6f} {(y2-y1)/720:.6f}")
            cls[MAP[c]] += 1
        boxes += len(lines)
        empty += (len(lines) == 0)
        open(f"{OUT}/labels/{sp}/{s}.txt", "w").write("\n".join(lines))
        a = d.get("attributes") or {}
        meta.append([sp, s, a.get("weather", ""), a.get("scene", ""), a.get("timeofday", "")])
    summary[sp] = (len(items), boxes, empty, cls)

open(f"{OUT}/data.yaml", "w").write(
    "path: /content/bdd_subset\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
os.makedirs(f"{BASE}/logs", exist_ok=True)
for p in (f"{OUT}/metadata.csv", f"{BASE}/logs/metadata.csv"):
    with open(p, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["split", "image", "weather", "scene", "timeofday"]); w.writerows(meta)

print("\n===== SUMMARY =====  (test fallback used: %s)" % fallback)
for sp, (ni, nb, ne, cls) in summary.items():
    print(f"{sp}: {ni} images | {nb} boxes | {ne} images with no boxes")
    print("   classes:", {NAMES[k]: v for k, v in sorted(cls.items())})
print("unknown categories skipped:", dict(unknown), "| tiny boxes skipped:", tiny)
print("weather:", collections.Counter(m[2] for m in meta).most_common(6))
print("time of day:", collections.Counter(m[4] for m in meta).most_common(6))
z = shutil.make_archive(f"{BASE}/data/bdd100k_subset", "zip", OUT)
print("Zip saved:", z, round(os.path.getsize(z) / 1e6, 1), "MB")

In [ ]:
import os, glob, collections, zipfile, csv
import matplotlib.pyplot as plt
from google.colab import files

BASE, OUT = "/content/drive/MyDrive/adp_project", "/content/bdd_subset"
assert os.path.isdir("/content/drive/MyDrive"), "Drive not mounted"
if not os.path.isdir(f"{OUT}/labels"):
    zipfile.ZipFile(f"{BASE}/data/bdd100k_subset.zip").extractall(OUT)

NAMES = ["person","rider","car","truck","bus","train","motor","bike","traffic light","traffic sign"]
counts = {sp: collections.Counter() for sp in ("train", "val", "test")}
for sp in counts:
    for p in glob.glob(f"{OUT}/labels/{sp}/*.txt"):
        for line in open(p):
            if line.strip():
                counts[sp][int(line.split()[0])] += 1

os.makedirs("/content/plots", exist_ok=True)
tr = [counts["train"][i] for i in range(10)]
tot = sum(tr)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(NAMES, tr, color="#2b6cb0")
ax.set_yscale("log"); ax.set_ylabel("boxes (log scale)")
ax.set_title("Class balance - training subset (4,000 images, 74,380 boxes)")
for b, v in zip(bars, tr):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v}\n{100*v/tot:.1f}%", ha="center", fontsize=8)
ax.set_ylim(5, max(tr) * 6)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("/content/plots/class_balance_train.png", dpi=150); plt.show()

rows = list(csv.DictReader(open(f"{BASE}/logs/metadata.csv")))
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in ((axs[0], "weather", "Weather"), (axs[1], "timeofday", "Time of day")):
    c = collections.Counter(r[key] or "missing" for r in rows).most_common()
    ax.bar([k for k, _ in c], [v for _, v in c], color="#2f855a")
    ax.set_title(f"{title} (all 5,600 images)"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("/content/plots/conditions.png", dpi=150); plt.show()

print("\nshare of boxes (train):", {NAMES[i]: f"{100*tr[i]/tot:.2f}%" for i in range(10)})
for f in ("/content/plots/class_balance_train.png", "/content/plots/conditions.png", f"{BASE}/logs/metadata.csv"):
    files.download(f)